# Exercise 0 - Cloud & Databricks Intro

## 0. More EDAs on Olympic data

In lecture 03 we ingested Olympic data from Kaggle and did very simple EDA. Now create more EDAs there.

  a) Start with reading in the dataset into a dataframe using spark.

  b) Use spark columns method to find out the columns

  c) Find out the 10 oldest atheletes, their age and the sport

  d) Find out the 10 youngest atheletes, their age and the sport

  e) Find out the five sports with highest median age

  f) Find out the five sports with lowest median age

  g) Find out top 10 countries after number of gold medals

  h) Find out top 10 countries after number of medals

  i) Plot a time series line chart of number of female and male atheletes in same graph.

  j) Do more explorations on your own

In [0]:
# a)
DATA_PATH = "/Volumes/data/olympic_games/raw_data"

df_athletes = spark.read.csv(f"{DATA_PATH}/athlete_events.csv", header=True)
df_athletes.display()

In [0]:
# b)

df_athletes.columns


In [0]:
from pyspark.sql.types import StructField, StringType, ShortType, ByteType, StructType, IntegerType, FloatType

schema = StructType([
  StructField("ID", IntegerType()),
  StructField("Name", StringType()),
  StructField("Sex", StringType()),
  StructField("Age", ByteType()),
  StructField("Height", FloatType()),
  StructField("Weight", FloatType()),
  StructField("Team", StringType()),
  StructField("NOC", StringType()),
  StructField("Games", StringType()),
  StructField("Year", ShortType()),
  StructField("Season", StringType()),
  StructField("City", StringType()),
  StructField("Sport", StringType()),
  StructField("Event", StringType()),
  StructField("Medal", StringType())
])

df_athletes_schema = spark.read.csv(f"{DATA_PATH}/athlete_events.csv", header=True, schema=schema)
display(df_athletes_schema)

In [0]:
# c)
df_athletes_schema.createOrReplaceTempView("df_athletes_schema")

oldest = spark.sql("""
          SELECT DISTINCT name, age, sport
          FROM df_athletes_schema
          ORDER BY age DESC 
          LIMIT 10
          """)

oldest.display()



In [0]:
# d)

youngest = spark.sql("""
          SELECT DISTINCT name, age, sport
          FROM df_athletes_schema
          WHERE age IS NOT NULL
          ORDER BY age ASC 
          LIMIT 10
          """)

youngest.display()


In [0]:
# e) 
top_median_age = spark.sql("""
          SELECT DISTINCT sport, 
          MEDIAN(age) as median_age
          FROM df_athletes_schema
          WHERE age IS NOT NULL
          GROUP BY sport
          ORDER BY median_age DESC
          LIMIT 5
          """)

top_median_age.display()


In [0]:
min_median_age = spark.sql("""
                SELECT DISTINCT sport,
                MEDIAN(age) as median_age_min
                FROM df_athletes_schema
                WHERE age IS NOT NULL
                GROUP BY sport
                ORDER BY median_age_min ASC
                LIMIT 5
                """)

min_median_age.display()

In [0]:
# g)
top_countries_gold = spark.sql("""
                SELECT NOC as country,
                COUNT(DISTINCT games, event, medal) as num_gold_medals
                FROM df_athletes_schema
                WHERE medal = 'Gold'
                GROUP BY NOC
                ORDER BY num_gold_medals DESC
                LIMIT 10
                """)

top_countries_gold.display()

In [0]:
# h)
top_countries_medals = spark.sql("""
                SELECT NOC as country,
                COUNT(DISTINCT games, event, medal) as num_medals
                FROM df_athletes_schema
                GROUP BY NOC
                ORDER BY num_medals DESC
                LIMIT 10
                """)

top_countries_medals.display()

In [0]:
# i)
num_athletes = spark.sql("""
                SELECT year, sex,
                COUNT(DISTINCT ID) as num_athletes
                FROM df_athletes_schema
                WHERE sex IN ('F', 'M')
                GROUP BY year, sex
                ORDER BY year DESC
                """)

#num_athletes.display()

# pivot to get columns 'F' & 'M'
num_athletes_plot = num_athletes.groupBy("year").pivot("sex").sum("num_athletes")

# create plot
fig = num_athletes_plot.plot(kind="line", x="year", y=["F", "M"], title="Number of athletes by year and gender")

# change x and y labels
# i)
num_athletes = spark.sql("""
                SELECT year, sex,
                COUNT(DISTINCT ID) as num_athletes
                FROM df_athletes_schema
                WHERE sex IN ('F', 'M')
                GROUP BY year, sex
                ORDER BY year DESC
                """)

#num_athletes.display()

num_athletes_plot = num_athletes.groupBy("year").pivot("sex").sum("num_athletes")

fig = num_athletes_plot.plot(kind="line", x="year", y=["F", "M"], title="Number of athletes by year and gender")

fig.update_layout(xaxis_title="Year", yaxis_title="Number of athletes")

display(fig)